# 02_bronze_fhir
Loads the Synthea FHIR R4 bundles into `health.bronze.fhir_bundles`, **one row per bundle file**, with the JSON kept as raw text, plus audit columns.

Patient files are `transaction` bundles; `hospitalInformation` and `practitionerInformation` are `batch` bundles.
Resources are exploded and flattened in silver.

In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F

files = [f.name for f in dbutils.fs.ls(landing(FHIR_VOLUME)) if f.name.endswith(".json")]
print(f"{len(files)} FHIR files waiting in landing")
if not files:
    dbutils.notebook.exit("No new FHIR files in landing; nothing to load.")

## Read the files
`wholetext` reads each file as a single row, so each bundle stays intact as one JSON document.
Inferring a schema across every FHIR resource type would be slow and fragile, so the JSON stays as text here.

In [0]:
df = (
    spark.read

        .text(landing(FHIR_VOLUME) + "*.json", wholetext=True)
        .select(
            F.col("value").alias("bundle_json"),
            F.get_json_object("value", "$.type").alias("bundle_type"),
            F.expr("json_array_length(get_json_object(value, '$.entry'))").alias("entry_count"),
            F.col("_metadata.file_name").alias("source_file"),
            F.col("_metadata.file_size").alias("source_file_size"),
            F.current_timestamp().alias("ingested_at"),
        )
)
display(df.drop("bundle_json").orderBy(F.desc("entry_count")).limit(5))
# Guard: one row per file. If this fails, the files were read line by line.
assert df.count() == len(files), f"Expected {len(files)} bundles, got {df.count()}"

## Write to bronze

In [0]:
(df.write
   .format("delta")
   .mode("append")
   .option("delta.enableChangeDataFeed", "true")
   .saveAsTable(f"{CATALOG}.{BRONZE}.fhir_bundles"))

move_to_processed(FHIR_VOLUME, files)

## Validate

In [0]:
%sql
-- Expect 114 bundles: 112 'transaction' (patients, 97,504 entries) and 2 'batch' (hospitals + practitioners, 1,069 entries)
SELECT bundle_type, COUNT(*) AS bundles, SUM(entry_count) AS entries
FROM health.bronze.fhir_bundles
GROUP BY bundle_type